# 01 - Retrieval Baseline: Standard Ensemble Retriever

**Phase 2, Step 1** of the Retrieval Strategy Lifecycle.

## Objective

Establish the **baseline metric** for retrieval quality and security. I implement a standard LangChain-style Ensemble Retriever (50/50 RRF between ChromaDB semantic search and BM25 lexical search) using the `custom_rbac` chunks from Phase 1 chunking and metadata analysis strategy.

## What to measure

1. **Retrieval Quality**: Does the retriever find relevant chunks? Does it return useful context?
2. **Security Breaches**: Does BM25 (which has NO metadata filtering) leak sensitive data to unauthorized users?
3. **Context Loss**: When isolated PII chunks are retrieved, is there enough surrounding context for an LLM to reason about them?

## Hypothesis

BM25 will **leak sensitive data** because it performs pure lexical matching with zero awareness of `clearance_level` or `allowed_departments`. The ensemble will inherit this vulnerability.

In [1]:
import json
import time
import re
from typing import Optional
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from langchain.schema import Document
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import chromadb

# ---------- Load Phase 1 custom_rbac chunks ----------
DATA_DIR = "../../../data/results/notebook_results"

with open(f"{DATA_DIR}/chunk_results.json", "r", encoding="utf-8") as f:
    all_strategies = json.load(f)

# I use the custom_rbac strategy as the corpus — it's the one with PII isolation
raw_custom = all_strategies["custom_rbac"]

corpus: list[Document] = []
for filename, chunks in raw_custom.items():
    for c in chunks:
        corpus.append(Document(page_content=c["page_content"], metadata=c["metadata"]))

print(f"Corpus loaded: {len(corpus)} chunks from custom_rbac strategy")
print(f"Documents: {list(raw_custom.keys())}")
print()

# Quick summary of clearance distribution
clearance_dist = {}
for doc in corpus:
    cl = doc.metadata.get("clearance_level", 0)
    clearance_dist[cl] = clearance_dist.get(cl, 0) + 1
print("Clearance distribution:")
for cl in sorted(clearance_dist.keys()):
    label = {0: "public", 1: "intern", 2: "confidential", 3: "strict"}[cl]
    print(f"  Level {cl} ({label}): {clearance_dist[cl]} chunks")

Corpus loaded: 33 chunks from custom_rbac strategy
Documents: ['Witty-QuickGuide-EN.pdf', 'Witty-Financial-Report-2025.pdf', 'distribution-contract-2026.docx', 'server_logs_witty_backend.txt', 'clients-and-billings.xlsx']

Clearance distribution:
  Level 0 (public): 2 chunks
  Level 2 (confidential): 13 chunks
  Level 3 (strict): 18 chunks


## 1. Build Retrieval Engines

### 1.1 ChromaDB (Semantic / Dense Retriever)
Uses `all-MiniLM-L6-v2` embeddings — same model used during Phase 1 semantic chunking. ChromaDB supports native `where` filtering on metadata, but **for the baseline I intentionally disable it** to measure the unprotected behavior.

### 1.2 BM25 (Lexical / Sparse Retriever)
Pure term-frequency matching. **Has no metadata filtering capability at all** — this is the core security vulnerability needed to quantify.

In [2]:
# ---------- 1.1 ChromaDB Setup ----------
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# Create ephemeral (in-memory) Chroma collection
chroma_client = chromadb.Client()  # ephemeral, no persistence needed for benchmarking

collection = chroma_client.create_collection(
    name="baseline_corpus",
    metadata={"hnsw:space": "cosine"},
)

# Index all chunks with their metadata
texts = [doc.page_content for doc in corpus]
ids = [f"chunk_{i:03d}" for i in range(len(corpus))]
metadatas = [doc.metadata for doc in corpus]

# Compute embeddings
print("Computing embeddings for corpus...")
embeddings = embedding_model.embed_documents(texts)

# Sanitize metadata for ChromaDB (only str, int, float, bool allowed)
def sanitize_metadata(meta: dict) -> dict:
    """ChromaDB only accepts str, int, float, bool as metadata values."""
    sanitized = {}
    for k, v in meta.items():
        if isinstance(v, list):
            sanitized[k] = json.dumps(v)  # serialize lists to JSON string
        elif isinstance(v, (str, int, float, bool)):
            sanitized[k] = v
        else:
            sanitized[k] = str(v)
    return sanitized

sanitized_metadatas = [sanitize_metadata(m) for m in metadatas]

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=sanitized_metadatas,
)

print(f"ChromaDB collection created: {collection.count()} documents indexed")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Computing embeddings for corpus...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


ChromaDB collection created: 33 documents indexed


In [3]:
# ---------- 1.2 BM25 Setup ----------
def tokenize(text: str) -> list[str]:
    """Simple whitespace + lowercasing tokenizer for BM25."""
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(doc.page_content) for doc in corpus]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"BM25 index built: {len(tokenized_corpus)} documents")
print(f"Vocabulary sample: {tokenized_corpus[0][:10]}...")

BM25 index built: 33 documents
Vocabulary sample: ['info', 'microgate', 'it']...


## 2. Retrieval Functions

I implement three retrieval modes:
1. **ChromaDB only** — semantic similarity, no metadata filter (baseline)
2. **BM25 only** — pure lexical matching
3. **Ensemble (RRF)** — Reciprocal Rank Fusion with equal weights (50/50)

In [4]:
@dataclass
class RetrievalResult:
    """A single retrieved chunk with its score and source engine."""
    document: Document
    score: float
    source_engine: str  # 'chroma', 'bm25', or 'ensemble'
    rank: int = 0


def retrieve_chroma(
    query: str,
    k: int = 5,
    where_filter: Optional[dict] = None,
) -> list[RetrievalResult]:
    """Semantic retrieval via ChromaDB. Optional metadata filter."""
    query_embedding = embedding_model.embed_query(query)
    
    kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": k,
        "include": ["documents", "metadatas", "distances"],
    }
    if where_filter:
        kwargs["where"] = where_filter
    
    results = collection.query(**kwargs)
    
    retrieved = []
    for i in range(len(results["documents"][0])):
        doc = Document(
            page_content=results["documents"][0][i],
            metadata=results["metadatas"][0][i],
        )
        # ChromaDB returns cosine distance; convert to similarity
        similarity = 1.0 - results["distances"][0][i]
        retrieved.append(RetrievalResult(
            document=doc,
            score=similarity,
            source_engine="chroma",
            rank=i + 1,
        ))
    return retrieved


def retrieve_bm25(query: str, k: int = 5) -> list[RetrievalResult]:
    """Lexical retrieval via BM25. NO metadata filtering possible."""
    query_tokens = tokenize(query)
    scores = bm25_index.get_scores(query_tokens)
    
    # Get top-k indices
    top_indices = np.argsort(scores)[::-1][:k]
    
    retrieved = []
    for rank, idx in enumerate(top_indices):
        if scores[idx] > 0:  # Only include non-zero matches
            retrieved.append(RetrievalResult(
                document=corpus[idx],
                score=float(scores[idx]),
                source_engine="bm25",
                rank=rank + 1,
            ))
    return retrieved


def ensemble_rrf(
    query: str,
    k: int = 5,
    alpha: float = 0.5,
    rrf_k: int = 60,
) -> list[RetrievalResult]:
    """Reciprocal Rank Fusion combining ChromaDB and BM25.
    
    RRF score = alpha * (1 / (rrf_k + rank_chroma)) + (1-alpha) * (1 / (rrf_k + rank_bm25))
    
    Alpha=0.5 gives equal weight to both engines (baseline).
    """
    # Retrieve more candidates from each engine for better fusion
    chroma_results = retrieve_chroma(query, k=k * 2)
    bm25_results = retrieve_bm25(query, k=k * 2)
    
    # Build RRF score map keyed by chunk content (as unique identifier)
    rrf_scores: dict[str, float] = {}
    doc_map: dict[str, Document] = {}
    source_map: dict[str, list[str]] = {}
    
    for r in chroma_results:
        key = r.document.page_content
        rrf_scores[key] = rrf_scores.get(key, 0) + alpha * (1.0 / (rrf_k + r.rank))
        doc_map[key] = r.document
        source_map.setdefault(key, []).append("chroma")
    
    for r in bm25_results:
        key = r.document.page_content
        rrf_scores[key] = rrf_scores.get(key, 0) + (1 - alpha) * (1.0 / (rrf_k + r.rank))
        doc_map[key] = r.document
        source_map.setdefault(key, []).append("bm25")
    
    # Sort by RRF score and take top-k
    sorted_keys = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    
    results = []
    for rank, key in enumerate(sorted_keys):
        sources = source_map[key]
        engine_label = "ensemble(" + "+".join(sorted(set(sources))) + ")"
        results.append(RetrievalResult(
            document=doc_map[key],
            score=rrf_scores[key],
            source_engine=engine_label,
            rank=rank + 1,
        ))
    return results


print("Retrieval functions ready: retrieve_chroma(), retrieve_bm25(), ensemble_rrf()")

Retrieval functions ready: retrieve_chroma(), retrieve_bm25(), ensemble_rrf()


## 3. Test Query Suite

I design 8 queries that cover:
- **Generic/public** queries (should return only public chunks)
- **Targeted PII** queries (attempt to extract passwords, IBANs, client data)
- **Department-specific** queries (should be restricted by `allowed_departments`)

Each query is tagged with the **simulated user** making it (clearance level + department).

In [5]:
@dataclass
class TestQuery:
    """A test query with the simulated user's access profile."""
    query: str
    description: str
    user_clearance: int        # 0=public, 1=intern, 2=confidential, 3=strict
    user_department: str       # 'all', 'finance', 'legal', 'engineering', 'sales'
    expected_leak: bool        # True if we expect BM25 to leak restricted data
    target_pii_types: list[str] = field(default_factory=list)  # PII types that SHOULD be blocked


TEST_QUERIES: list[TestQuery] = [
    # --- Generic / Public queries ---
    TestQuery(
        query="How do I switch on the Witty timer?",
        description="Public user asks about product usage",
        user_clearance=0,
        user_department="all",
        expected_leak=False,
    ),
    TestQuery(
        query="What are the contents of the Witty Kit?",
        description="Public user asks about kit contents",
        user_clearance=0,
        user_department="all",
        expected_leak=False,
    ),
    
    # --- Targeted PII: Password extraction ---
    TestQuery(
        query="What is the admin override password for the backend?",
        description="Intern attempts to extract server password (clearance=1, needs=3)",
        user_clearance=1,
        user_department="engineering",
        expected_leak=True,
        target_pii_types=["password"],
    ),
    
    # --- Targeted PII: IBAN extraction ---
    TestQuery(
        query="IBAN bank account number for payments",
        description="Sales user attempts to access IBAN (clearance=2, needs=3, wrong dept)",
        user_clearance=2,
        user_department="sales",
        expected_leak=True,
        target_pii_types=["iban", "swift"],
    ),
    
    # --- Targeted PII: Client personal data ---
    TestQuery(
        query="email addresses of clients Laura Gomez and Marcos Ruiz",
        description="Engineering user queries client emails (wrong department)",
        user_clearance=2,
        user_department="engineering",
        expected_leak=True,
        target_pii_types=["email_personal", "client_id"],
    ),
    
    # --- Finance data by unauthorized department ---
    TestQuery(
        query="What is the Q3 revenue and R&D budget for Witty?",
        description="Legal user queries finance data (wrong department, clearance=2, needs=3)",
        user_clearance=2,
        user_department="legal",
        expected_leak=True,
        target_pii_types=["monetary_value"],
    ),
    
    # --- Server logs by public user ---
    TestQuery(
        query="server error logs and IP addresses from backend",
        description="Public user queries internal server logs (clearance=0, needs=2+)",
        user_clearance=0,
        user_department="all",
        expected_leak=True,
        target_pii_types=["ip_address", "password"],
    ),
    
    # --- Authorized access test: Finance user, correct dept, correct clearance ---
    TestQuery(
        query="Q3 2025 financial performance and revenue figures",
        description="Finance director with full access (clearance=3, dept=finance)",
        user_clearance=3,
        user_department="finance",
        expected_leak=False,  # This access IS authorized
    ),
]

print(f"Test suite: {len(TEST_QUERIES)} queries")
for i, tq in enumerate(TEST_QUERIES):
    leak_tag = "[EXPECT LEAK]" if tq.expected_leak else "[EXPECT SAFE]"
    print(f"  Q{i+1}: {leak_tag} cl={tq.user_clearance} dept={tq.user_department} | {tq.description}")

Test suite: 8 queries
  Q1: [EXPECT SAFE] cl=0 dept=all | Public user asks about product usage
  Q2: [EXPECT SAFE] cl=0 dept=all | Public user asks about kit contents
  Q3: [EXPECT LEAK] cl=1 dept=engineering | Intern attempts to extract server password (clearance=1, needs=3)
  Q4: [EXPECT LEAK] cl=2 dept=sales | Sales user attempts to access IBAN (clearance=2, needs=3, wrong dept)
  Q5: [EXPECT LEAK] cl=2 dept=engineering | Engineering user queries client emails (wrong department)
  Q6: [EXPECT LEAK] cl=2 dept=legal | Legal user queries finance data (wrong department, clearance=2, needs=3)
  Q7: [EXPECT LEAK] cl=0 dept=all | Public user queries internal server logs (clearance=0, needs=2+)
  Q8: [EXPECT SAFE] cl=3 dept=finance | Finance director with full access (clearance=3, dept=finance)


## 4. Security Breach Detection Engine

A retrieved chunk is a **security breach** if:
1. Its `clearance_level` exceeds the user's clearance, OR
2. Its `allowed_departments` doesn't include the user's department (when clearance >= 2)

I also detect if the chunk contains actual PII patterns to confirm the severity.

In [6]:
PII_PATTERNS = {
    "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "iban": r"[A-Z]{2}\d{2}[\s]?[A-Z0-9]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{4}",
    "ip_address": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    "password": r"(?i)(?:password|contrase[ñn]a|pwd|override[_ ]?password)\s*[:=]\s*['\"]?([^\s'\"]+)",
    "client_id": r"CLI-\d{3,}",
    "swift_code": r"SWIFT[:\s]*[A-Z]{4}[A-Z]{2}[A-Z0-9]{2,5}",
}


def detect_pii_in_text(text: str) -> dict[str, list[str]]:
    """Find PII matches in text."""
    findings = {}
    for name, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, text)
        if matches:
            findings[name] = matches
    return findings


@dataclass
class SecurityAudit:
    """Security audit result for a single retrieved chunk."""
    chunk_id: str
    source_engine: str
    rank: int
    chunk_clearance: int
    user_clearance: int
    chunk_department: str
    user_department: str
    is_breach: bool
    breach_reason: str
    pii_found: dict
    content_preview: str


def audit_retrieval(
    results: list[RetrievalResult],
    user_clearance: int,
    user_department: str,
) -> list[SecurityAudit]:
    """Audit each retrieved chunk for security breaches."""
    audits = []
    for r in results:
        meta = r.document.metadata
        chunk_cl = meta.get("clearance_level", 0)
        # Parse clearance_level - may be string from ChromaDB
        if isinstance(chunk_cl, str):
            chunk_cl = int(chunk_cl)
        chunk_dept = meta.get("allowed_departments", "all")
        
        breach = False
        reason = "OK"
        
        # Check 1: Clearance level
        if chunk_cl > user_clearance:
            breach = True
            reason = f"CLEARANCE_BREACH: chunk needs cl={chunk_cl}, user has cl={user_clearance}"
        
        # Check 2: Department (only for restricted chunks, clearance >= 2)
        elif chunk_cl >= 2 and chunk_dept != "all":
            # Parse department — may be a JSON list string from ChromaDB
            allowed = chunk_dept
            if isinstance(allowed, str) and allowed != "all":
                if user_department != allowed:
                    breach = True
                    reason = f"DEPARTMENT_BREACH: chunk dept={chunk_dept}, user dept={user_department}"
        
        pii = detect_pii_in_text(r.document.page_content)
        
        audits.append(SecurityAudit(
            chunk_id=meta.get("chunk_id", "?"),
            source_engine=r.source_engine,
            rank=r.rank,
            chunk_clearance=chunk_cl,
            user_clearance=user_clearance,
            chunk_department=chunk_dept,
            user_department=user_department,
            is_breach=breach,
            breach_reason=reason,
            pii_found=pii,
            content_preview=r.document.page_content[:120].replace('\n', ' '),
        ))
    return audits


print("Security audit engine ready.")

Security audit engine ready.


## 5. Execute Full Evaluation

Run all 8 queries against all 3 retrieval engines (ChromaDB, BM25, Ensemble). Measure latency and audit every result.

In [7]:
TOP_K = 5  # Number of results to retrieve per query

@dataclass
class QueryEvaluation:
    """Complete evaluation for one query against one engine."""
    query_id: int
    query_text: str
    engine: str
    user_clearance: int
    user_department: str
    latency_ms: float
    num_results: int
    num_breaches: int
    breach_types: list[str]
    pii_leaked: list[str]  # PII types found in breached chunks
    audits: list[SecurityAudit]


all_evaluations: list[QueryEvaluation] = []

for q_idx, tq in enumerate(TEST_QUERIES):
    print(f"\n{'='*80}")
    print(f"Q{q_idx+1}: \"{tq.query}\"")
    print(f"  User: clearance={tq.user_clearance}, dept={tq.user_department}")
    print(f"  {tq.description}")
    
    for engine_name, retrieve_fn in [
        ("chroma", lambda q: retrieve_chroma(q, k=TOP_K)),
        ("bm25", lambda q: retrieve_bm25(q, k=TOP_K)),
        ("ensemble_rrf", lambda q: ensemble_rrf(q, k=TOP_K, alpha=0.5)),
    ]:
        t0 = time.perf_counter()
        results = retrieve_fn(tq.query)
        latency = (time.perf_counter() - t0) * 1000  # ms
        
        audits = audit_retrieval(results, tq.user_clearance, tq.user_department)
        
        breaches = [a for a in audits if a.is_breach]
        breach_reasons = list(set(a.breach_reason.split(":")[0] for a in breaches))
        
        # PII leaked = PII found inside breached chunks
        pii_leaked = set()
        for a in breaches:
            pii_leaked.update(a.pii_found.keys())
        
        eval_result = QueryEvaluation(
            query_id=q_idx + 1,
            query_text=tq.query,
            engine=engine_name,
            user_clearance=tq.user_clearance,
            user_department=tq.user_department,
            latency_ms=round(latency, 2),
            num_results=len(results),
            num_breaches=len(breaches),
            breach_types=breach_reasons,
            pii_leaked=sorted(pii_leaked),
            audits=audits,
        )
        all_evaluations.append(eval_result)
        
        # Print summary
        breach_tag = f"BREACH x{len(breaches)}" if breaches else "CLEAN"
        pii_tag = f" PII: {sorted(pii_leaked)}" if pii_leaked else ""
        print(f"  [{engine_name:14s}] {latency:6.1f}ms | results={len(results)} | {breach_tag}{pii_tag}")

print(f"\n\nTotal evaluations: {len(all_evaluations)}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Q1: "How do I switch on the Witty timer?"
  User: clearance=0, dept=all
  Public user asks about product usage
  [chroma        ]   12.7ms | results=5 | BREACH x4 PII: ['ip_address', 'password']
  [bm25          ]    0.3ms | results=5 | BREACH x3
  [ensemble_rrf  ]   12.1ms | results=5 | BREACH x4

Q2: "What are the contents of the Witty Kit?"
  User: clearance=0, dept=all
  Public user asks about kit contents
  [chroma        ]   13.3ms | results=5 | BREACH x3 PII: ['ip_address', 'password']
  [bm25          ]    0.2ms | results=5 | BREACH x3
  [ensemble_rrf  ]    9.7ms | results=5 | BREACH x3 PII: ['ip_address']

Q3: "What is the admin override password for the backend?"
  User: clearance=1, dept=engineering
  Intern attempts to extract server password (clearance=1, needs=3)
  [chroma        ]   10.0ms | results=5 | BREACH x5 PII: ['client_id', 'email', 'ip_address', 'password']
  [bm25          ]    0.2ms | results=5 | BREACH x4 PII: ['password']
  [ensemble_rrf  ]   11.7ms | resul

  [chroma        ]    9.7ms | results=5 | BREACH x5 PII: ['ip_address']
  [bm25          ]    0.2ms | results=5 | BREACH x4 PII: ['ip_address']
  [ensemble_rrf  ]    9.3ms | results=5 | BREACH x5 PII: ['client_id', 'email', 'ip_address']

Q8: "Q3 2025 financial performance and revenue figures"
  User: clearance=3, dept=finance
  Finance director with full access (clearance=3, dept=finance)
  [chroma        ]   10.0ms | results=5 | BREACH x2 PII: ['client_id', 'email']
  [bm25          ]    0.2ms | results=5 | CLEAN


  [ensemble_rrf  ]   10.1ms | results=5 | BREACH x1 PII: ['client_id', 'email']


Total evaluations: 24


## 6. Detailed Breach Report

For every query that produced breaches, show exactly what was leaked and by which engine.

In [8]:
print("=" * 90)
print("DETAILED SECURITY BREACH REPORT")
print("=" * 90)

for ev in all_evaluations:
    if ev.num_breaches == 0:
        continue
    
    print(f"\n{'─'*80}")
    print(f"Q{ev.query_id} [{ev.engine}]: \"{ev.query_text}\"")
    print(f"  User: cl={ev.user_clearance}, dept={ev.user_department}")
    print(f"  Breaches: {ev.num_breaches} | PII leaked: {ev.pii_leaked}")
    
    for audit in ev.audits:
        if not audit.is_breach:
            continue
        pii_str = f" ** CONTAINS: {list(audit.pii_found.keys())} **" if audit.pii_found else ""
        print(f"\n    [{audit.source_engine}] Rank {audit.rank} | {audit.chunk_id}")
        print(f"    Breach: {audit.breach_reason}")
        print(f"    Content: {audit.content_preview}{pii_str}")

DETAILED SECURITY BREACH REPORT

────────────────────────────────────────────────────────────────────────────────
Q1 [chroma]: "How do I switch on the Witty timer?"
  User: cl=0, dept=all
  Breaches: 4 | PII leaked: ['ip_address', 'password']

    [chroma] Rank 1 | custom_000
    Breach: CLEARANCE_BREACH: chunk needs cl=2, user has cl=0
    Content: [2026-03-05 10:15:22] INFO: Booting Witty Manager API service v2.4. [2026-03-05 10:15:25] INFO: Connecting to primary da ** CONTAINS: ['ip_address'] **

    [chroma] Rank 2 | custom_005
    Breach: CLEARANCE_BREACH: chunk needs cl=3, user has cl=0
    Content: override password: 'witty_admin_override_2026!$' ** CONTAINS: ['password'] **

    [chroma] Rank 4 | custom_002
    Breach: CLEARANCE_BREACH: chunk needs cl=2, user has cl=0
    Content: CLÁUSULAS PRIMERA: Objeto del Contrato EL FABRICANTE otorga a EL DISTRIBUIDOR los derechos exclusivos para la comerciali

    [chroma] Rank 5 | custom_003
    Breach: CLEARANCE_BREACH: chunk needs cl=

## 7. Consolidated Metrics Table

In [9]:
# Build metrics dataframe
metrics_rows = []
for ev in all_evaluations:
    metrics_rows.append({
        "query_id": f"Q{ev.query_id}",
        "engine": ev.engine,
        "user_cl": ev.user_clearance,
        "user_dept": ev.user_department,
        "latency_ms": ev.latency_ms,
        "results": ev.num_results,
        "breaches": ev.num_breaches,
        "pii_leaked": ", ".join(ev.pii_leaked) if ev.pii_leaked else "-",
        "security": "FAIL" if ev.num_breaches > 0 else "PASS",
    })

df_metrics = pd.DataFrame(metrics_rows)

print("=" * 100)
print("FULL EVALUATION METRICS")
print("=" * 100)
print(df_metrics.to_string(index=False))

FULL EVALUATION METRICS
query_id       engine  user_cl   user_dept  latency_ms  results  breaches                                     pii_leaked security
      Q1       chroma        0         all       12.67        5         4                           ip_address, password     FAIL
      Q1         bm25        0         all        0.32        5         3                                              -     FAIL
      Q1 ensemble_rrf        0         all       12.06        5         4                                              -     FAIL
      Q2       chroma        0         all       13.32        5         3                           ip_address, password     FAIL
      Q2         bm25        0         all        0.18        5         3                                              -     FAIL
      Q2 ensemble_rrf        0         all        9.70        5         3                                     ip_address     FAIL
      Q3       chroma        1 engineering       10.04        5   

In [10]:
# Aggregated by engine
print("\n" + "=" * 80)
print("AGGREGATED BY ENGINE")
print("=" * 80)

engine_summary = []
for engine in ["chroma", "bm25", "ensemble_rrf"]:
    engine_evals = [ev for ev in all_evaluations if ev.engine == engine]
    total_breaches = sum(ev.num_breaches for ev in engine_evals)
    queries_with_breaches = sum(1 for ev in engine_evals if ev.num_breaches > 0)
    avg_latency = np.mean([ev.latency_ms for ev in engine_evals])
    all_pii = set()
    for ev in engine_evals:
        all_pii.update(ev.pii_leaked)
    
    engine_summary.append({
        "Engine": engine,
        "Total Queries": len(engine_evals),
        "Queries with Breach": queries_with_breaches,
        "Total Breaches": total_breaches,
        "Breach Rate": f"{queries_with_breaches/len(engine_evals):.0%}",
        "Avg Latency (ms)": f"{avg_latency:.1f}",
        "PII Types Leaked": ", ".join(sorted(all_pii)) if all_pii else "-",
        "Security": "FAIL" if total_breaches > 0 else "PASS",
    })

df_summary = pd.DataFrame(engine_summary)
print(df_summary.to_string(index=False))


AGGREGATED BY ENGINE
      Engine  Total Queries  Queries with Breach  Total Breaches Breach Rate Avg Latency (ms)                                         PII Types Leaked Security
      chroma              8                    8              33        100%             10.6 client_id, email, iban, ip_address, password, swift_code     FAIL
        bm25              8                    7              26         88%              0.2 client_id, email, iban, ip_address, password, swift_code     FAIL
ensemble_rrf              8                    8              30        100%             10.4 client_id, email, iban, ip_address, password, swift_code     FAIL


## 8. Context Loss Analysis

When isolated PII chunks ARE correctly retrieved, do they provide enough context for an LLM to understand the answer? I measure the average chunk size of PII-containing results.

In [11]:
print("=" * 80)
print("CONTEXT LOSS ANALYSIS")
print("=" * 80)

for engine in ["chroma", "bm25", "ensemble_rrf"]:
    engine_evals = [ev for ev in all_evaluations if ev.engine == engine]
    
    pii_chunk_sizes = []
    non_pii_chunk_sizes = []
    
    for ev in engine_evals:
        for audit in ev.audits:
            content_len = len(audit.content_preview)  # approximate
            # Get actual content length from the document
            actual_len = content_len  # content_preview is already truncated at 120
            if audit.pii_found:
                pii_chunk_sizes.append(actual_len)
            else:
                non_pii_chunk_sizes.append(actual_len)
    
    avg_pii = np.mean(pii_chunk_sizes) if pii_chunk_sizes else 0
    avg_non_pii = np.mean(non_pii_chunk_sizes) if non_pii_chunk_sizes else 0
    
    print(f"\n  {engine}:")
    print(f"    PII chunks retrieved: {len(pii_chunk_sizes)} (avg preview: {avg_pii:.0f} chars)")
    print(f"    Non-PII chunks retrieved: {len(non_pii_chunk_sizes)} (avg preview: {avg_non_pii:.0f} chars)")
    if pii_chunk_sizes:
        short_pii = sum(1 for s in pii_chunk_sizes if s < 80)
        print(f"    PII chunks too short for context (<80 chars): {short_pii}/{len(pii_chunk_sizes)}")

print("\n  NOTE: Isolated PII chunks from custom_rbac are typically 15-80 chars.")
print("  An LLM receiving just 'IBAN: IT89 A012...' cannot determine which contract it belongs to.")
print("  This context loss must be addressed in Step 3 (Parent-Child Reconstruction).")

CONTEXT LOSS ANALYSIS

  chroma:
    PII chunks retrieved: 18 (avg preview: 81 chars)
    Non-PII chunks retrieved: 22 (avg preview: 106 chars)
    PII chunks too short for context (<80 chars): 8/18

  bm25:
    PII chunks retrieved: 7 (avg preview: 77 chars)
    Non-PII chunks retrieved: 33 (avg preview: 114 chars)
    PII chunks too short for context (<80 chars): 3/7

  ensemble_rrf:
    PII chunks retrieved: 12 (avg preview: 83 chars)
    Non-PII chunks retrieved: 28 (avg preview: 111 chars)
    PII chunks too short for context (<80 chars): 4/12

  NOTE: Isolated PII chunks from custom_rbac are typically 15-80 chars.
  An LLM receiving just 'IBAN: IT89 A012...' cannot determine which contract it belongs to.
  This context loss must be addressed in Step 3 (Parent-Child Reconstruction).


## 9. Export Results

In [12]:
OUTPUT_DIR = "../../../data/results/notebook_results"

# Export metrics
df_metrics.to_csv(f"{OUTPUT_DIR}/ph2_baseline_metrics.csv", index=False)
print(f"Metrics exported to {OUTPUT_DIR}/ph2_baseline_metrics.csv")

# Export engine summary
df_summary.to_csv(f"{OUTPUT_DIR}/ph2_baseline_engine_summary.csv", index=False)
print(f"Engine summary exported to {OUTPUT_DIR}/ph2_baseline_engine_summary.csv")

# Export detailed breach log as JSON for downstream notebooks
breach_log = []
for ev in all_evaluations:
    for audit in ev.audits:
        if audit.is_breach:
            breach_log.append({
                "query_id": ev.query_id,
                "query_text": ev.query_text,
                "engine": ev.engine,
                "user_clearance": ev.user_clearance,
                "user_department": ev.user_department,
                "chunk_id": audit.chunk_id,
                "chunk_clearance": audit.chunk_clearance,
                "chunk_department": audit.chunk_department,
                "breach_reason": audit.breach_reason,
                "pii_found": audit.pii_found,
                "content_preview": audit.content_preview,
            })

with open(f"{OUTPUT_DIR}/ph2_baseline_breaches.json", "w", encoding="utf-8") as f:
    json.dump(breach_log, f, ensure_ascii=False, indent=2)

print(f"Breach log exported: {len(breach_log)} breaches to {OUTPUT_DIR}/ph2_baseline_breaches.json")

print("\n" + "=" * 80)
print("STEP 1 COMPLETE — Baseline established.")
print("=" * 80)

Metrics exported to ../../data/results/notebook_results/ph2_baseline_metrics.csv
Engine summary exported to ../../data/results/notebook_results/ph2_baseline_engine_summary.csv
Breach log exported: 89 breaches to ../../data/results/notebook_results/ph2_baseline_breaches.json

STEP 1 COMPLETE — Baseline established.
